# TankBind 虚拟筛选教程

**论文**: Lu Wei et al., *TankBind: Trigonometry-Aware Neural BINDing for Target-Aware Drug Design via Protein-Ligand Binding Structure Prediction*, NeurIPS 2022.

## 核心创新

TankBind 的核心思想是将分子对接问题转化为 **蛋白-配体距离矩阵预测** 问题：

1. 将蛋白质口袋原子和配体原子分别嵌入隐空间；
2. 构建蛋白-配体原子对的 **距离矩阵** 表示 $z_{ij}$；
3. 通过 **三角更新模块** (Triangle Update) 增强几何一致性 -- 借鉴自 AlphaFold2 的三角乘法更新思想；
4. 预测蛋白-配体距离矩阵，再用 **梯度下降** 从距离矩阵恢复配体 3D 坐标。

## 与传统打分模型的区别

| 模型类型 | 预测目标 | 输出 |
|---------|---------|------|
| 打分模型 (IGN/PIGNet/RTMScore) | 结合亲和力 | 标量 $\log K_a$ |
| **TankBind (对接模型)** | **空间几何信息** | **$N_p \times N_l$ 距离矩阵** |

## 学习目标

通过本教程，你将学习：
- 对接 (docking) 任务如何将目标从标量预测转化为距离矩阵预测
- 三角更新模块如何利用 $z_{ik}$ 与 $z_{kj}$ 的乘积来传播几何约束
- 如何通过梯度下降从预测距离矩阵恢复配体 3D 坐标
- RMSD 作为对接质量评估指标的含义

In [ ]:
# ============================================================
# 环境设置、路径配置与依赖导入
# ============================================================

def find_project_root(marker='demo_data'):
    """向上查找包含 demo_data/ 的项目根目录"""
    from pathlib import Path
    here = Path('.').resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'无法找到包含 {marker}/ 的项目根目录')

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'demo_data'
CORESET_FILE = DATA_DIR / 'CoreSet.dat'
COMPLEX_DIR = DATA_DIR / 'coreset'

import os
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit import RDLogger
from scipy.spatial import distance_matrix
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

%matplotlib inline

# 抑制 RDKit 的冗余警告信息
RDLogger.logger().setLevel(RDLogger.ERROR)
warnings.filterwarnings('ignore')

# 版本信息
version_info = pd.DataFrame({
    '库': ['torch', 'rdkit', 'numpy', 'scipy', 'pandas', 'matplotlib'],
    '版本': [
        torch.__version__,
        Chem.rdBase.rdkitVersion,
        np.__version__,
        __import__('scipy').__version__,
        pd.__version__,
        __import__('matplotlib').__version__,
    ]
})
display(version_info)

print(f"\n项目根目录: {PROJECT_ROOT}")
print(f"数据目录:   {DATA_DIR}")

## 1. 超参数设置

TankBind 对接模型的关键超参数包括：
- **距离矩阵相关**: `MAX_DIST` 控制距离截断上限，`POCKET_CUTOFF` 用于截取口袋原子
- **三角更新**: `N_TRIANGLE_LAYERS` 控制三角更新层数
- **坐标恢复**: `COORD_RECOVERY_STEPS` 和 `COORD_RECOVERY_LR` 控制梯度下降坐标恢复过程

In [ ]:
# ---------- 超参数 ----------
ATOM_FEAT_DIM = 10          # 简化原子特征维度
HIDDEN_DIM = 128            # 隐层维度
N_EPOCHS = 200              # 训练轮数
LR = 1e-3                   # 学习率
BATCH_SIZE = 1              # 变长结构，逐样本处理
SEED = 42
MAX_DIST = 20.0             # 距离矩阵预测上限 (Angstrom)
POCKET_CUTOFF = 8.0         # 用于截取口袋原子的距离阈值 (Angstrom)
N_TRIANGLE_LAYERS = 3       # 三角更新层数
COORD_RECOVERY_STEPS = 2000 # 坐标恢复梯度下降步数
COORD_RECOVERY_LR = 0.1    # 坐标恢复学习率（参考 TankBind 原始实现）
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

# 展示超参数表格
params_df = pd.DataFrame({
    '参数': ['ATOM_FEAT_DIM', 'HIDDEN_DIM', 'N_EPOCHS', 'LR', 'BATCH_SIZE',
            'MAX_DIST', 'POCKET_CUTOFF', 'N_TRIANGLE_LAYERS',
            'COORD_RECOVERY_STEPS', 'COORD_RECOVERY_LR', 'DEVICE'],
    '值': [ATOM_FEAT_DIM, HIDDEN_DIM, N_EPOCHS, LR, BATCH_SIZE,
           f'{MAX_DIST} A', f'{POCKET_CUTOFF} A', N_TRIANGLE_LAYERS,
           COORD_RECOVERY_STEPS, COORD_RECOVERY_LR, str(DEVICE)],
    '说明': ['简化原子特征维度', '隐层维度', '训练轮数', '学习率',
            '变长结构逐样本处理', '距离矩阵预测上限', '口袋原子截取阈值',
            '三角更新层数', '坐标恢复梯度下降步数', '坐标恢复学习率（参考原始实现）',
            '计算设备']
})
display(params_df)

## 2. 数据加载与特征提取

数据处理流程：
1. 从 `CoreSet.dat` 解析 PDB ID 列表（TankBind 是对接模型，`logKa` 仅用于获取有效 PDB ID）
2. 对每个复合物，加载蛋白口袋 (`.pdb`) 和配体 (`.mol2`/`.sdf`)
3. 提取 10 维原子特征（8 种常见元素 one-hot + 1 other + 1 芳香性）
4. 截取离配体较近的口袋原子，降低计算量
5. 计算蛋白-配体真实距离矩阵（截断到 `MAX_DIST`）

In [ ]:
# ---- 元素符号 -> one-hot 映射 (9类 + 1 芳香性 = 10维) ----
ELEMENT_LIST = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br']  # 8 种常见元素 + 1 other


def parse_coreset(path):
    """解析 CoreSet.dat，返回 {pdbid: logKa} 字典。跳过以 '#' 开头的注释行。
    注意：TankBind 是对接模型，logKa 仅用于获取有效 PDB ID 列表。"""
    labels = {}
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            pdbid = parts[0]
            logKa = float(parts[3])
            labels[pdbid] = logKa
    print(f"[INFO] 从 CoreSet.dat 读取到 {len(labels)} 个复合物")
    return labels


def atom_features(atom):
    """
    构建 10 维原子特征向量：
      - 前 9 维: 元素类型 one-hot (C, N, O, S, F, P, Cl, Br, other)
      - 第 10 维: 是否为芳香性原子
    """
    feat = np.zeros(ATOM_FEAT_DIM, dtype=np.float32)
    symbol = atom.GetSymbol()
    if symbol in ELEMENT_LIST:
        feat[ELEMENT_LIST.index(symbol)] = 1.0
    else:
        feat[8] = 1.0       # other 类别
    feat[9] = float(atom.GetIsAromatic())
    return feat


def load_mol(path, fmt):
    """
    用 RDKit 加载分子文件。
    fmt: 'mol2', 'sdf', 'pdb'
    先尝试正常加载，再尝试 sanitize=False 后手动 sanitize。
    """
    mol = None
    if fmt == 'mol2':
        mol = Chem.MolFromMol2File(path, sanitize=True)
        if mol is None:
            mol = Chem.MolFromMol2File(path, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass
    elif fmt == 'sdf':
        supplier = Chem.SDMolSupplier(path, sanitize=True)
        for m in supplier:
            if m is not None:
                mol = m
                break
        if mol is None:
            supplier = Chem.SDMolSupplier(path, sanitize=False)
            for m in supplier:
                if m is not None:
                    mol = m
                    try:
                        Chem.SanitizeMol(mol)
                    except Exception:
                        pass
                    break
    elif fmt == 'pdb':
        mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=True)
        if mol is None:
            mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass
    return mol


def compute_rmsd(pred_coords, true_coords):
    """计算两组坐标之间的 RMSD (均方根偏差)。
    pred_coords, true_coords: (N, 3) numpy 数组
    公式: RMSD = sqrt( mean( sum_i (pred_i - true_i)^2 ) )
    """
    diff = pred_coords - true_coords
    return np.sqrt(np.mean(np.sum(diff ** 2, axis=1)))


def build_tankbind_data(pdbid):
    """
    为单个蛋白-配体复合物构建 TankBind 数据。

    返回:
      prot_feats  : (N_p, 10)   蛋白口袋原子特征
      lig_feats   : (N_l, 10)   配体原子特征
      prot_coords : (N_p, 3)    蛋白口袋原子坐标
      lig_coords  : (N_l, 3)    配体原子坐标 (晶体结构真实坐标)
      dist_matrix : (N_p, N_l)  蛋白-配体真实距离矩阵
    如果解析失败则返回 None。
    """
    pocket_path = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_pocket.pdb")
    ligand_mol2 = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_ligand.mol2")
    ligand_sdf = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_ligand.sdf")

    # ---- 1. 加载蛋白口袋 ----
    prot_mol = Chem.MolFromPDBFile(pocket_path, removeHs=True, sanitize=False)
    if prot_mol is None:
        return None
    try:
        Chem.SanitizeMol(prot_mol)
    except Exception:
        pass

    # ---- 2. 加载配体 (mol2 优先, sdf 备选) ----
    lig_mol = Chem.MolFromMol2File(ligand_mol2, sanitize=False)
    if lig_mol is None and os.path.exists(ligand_sdf):
        lig_mol = load_mol(ligand_sdf, 'sdf')
    if lig_mol is None:
        return None
    try:
        Chem.SanitizeMol(lig_mol)
    except Exception:
        pass

    # ---- 3. 去氢 ----
    try:
        prot_mol = Chem.RemoveHs(prot_mol)
    except Exception:
        pass
    try:
        lig_mol = Chem.RemoveHs(lig_mol)
    except Exception:
        pass

    # ---- 4. 提取 3D 坐标与原子特征 ----
    prot_conf = prot_mol.GetConformer()
    lig_conf = lig_mol.GetConformer()

    prot_coords = np.array([prot_conf.GetAtomPosition(i)
                            for i in range(prot_mol.GetNumAtoms())], dtype=np.float32)
    lig_coords = np.array([lig_conf.GetAtomPosition(i)
                           for i in range(lig_mol.GetNumAtoms())], dtype=np.float32)

    prot_feats = np.array([atom_features(a) for a in prot_mol.GetAtoms()], dtype=np.float32)
    lig_feats = np.array([atom_features(a) for a in lig_mol.GetAtoms()], dtype=np.float32)

    if prot_feats.shape[0] == 0 or lig_feats.shape[0] == 0:
        return None

    # ---- 5. 截取离配体较近的口袋原子，降低计算量 ----
    # 计算每个蛋白原子到配体质心的距离，只保留 POCKET_CUTOFF 内的原子
    lig_center = lig_coords.mean(axis=0)
    prot_dists_to_center = np.linalg.norm(prot_coords - lig_center, axis=1)
    pocket_mask = prot_dists_to_center < POCKET_CUTOFF
    if pocket_mask.sum() < 3:
        # 如果口袋内原子太少，取最近的 20 个
        top_k = min(20, len(prot_coords))
        pocket_idx = np.argsort(prot_dists_to_center)[:top_k]
        prot_coords = prot_coords[pocket_idx]
        prot_feats = prot_feats[pocket_idx]
    else:
        prot_coords = prot_coords[pocket_mask]
        prot_feats = prot_feats[pocket_mask]

    # ---- 6. 计算蛋白-配体真实距离矩阵 ----
    # dist_mat[i, j] = 蛋白原子 i 到配体原子 j 的欧氏距离
    dist_mat = distance_matrix(prot_coords, lig_coords).astype(np.float32)

    # 参考 TankBind 原始实现：将距离截断到 MAX_DIST 内
    # 远距离原子对的精确距离对对接无意义，截断可降低噪声
    dist_mat = np.clip(dist_mat, 0, MAX_DIST)

    return prot_feats, lig_feats, prot_coords, lig_coords, dist_mat

In [ ]:
# ---- 预处理所有复合物 ----
print("正在构建 TankBind 数据...")
labels = parse_coreset(str(CORESET_FILE))

all_data = []
failed = 0
for pdbid in sorted(labels.keys()):
    result = build_tankbind_data(pdbid)
    if result is None:
        failed += 1
        continue
    all_data.append(result)

print(f"[INFO] 成功加载 {len(all_data)} 个复合物, 失败 {failed} 个")

# ---- 展示一个样本的数据形状 ----
sample = all_data[0]
sample_info = pd.DataFrame({
    '数据': ['prot_feats', 'lig_feats', 'prot_coords', 'lig_coords', 'dist_matrix'],
    '形状': [str(sample[0].shape), str(sample[1].shape), str(sample[2].shape),
            str(sample[3].shape), str(sample[4].shape)],
    '说明': ['蛋白口袋原子特征', '配体原子特征', '蛋白口袋原子坐标',
            '配体原子坐标 (真实)', '蛋白-配体真实距离矩阵']
})
display(sample_info)

## 3. 数据集与数据加载器

将预处理好的数据封装为 PyTorch `Dataset` 和 `DataLoader`。

注意：
- 由于每个复合物的蛋白/配体原子数不同（变长结构），`BATCH_SIZE=1`
- 训练目标是**距离矩阵**（而非 `logKa`），这是对接任务与打分任务的根本区别

In [ ]:
class TankBindDataset(Dataset):
    """TankBind 数据集。每个样本包含蛋白/配体特征、坐标和真实距离矩阵。
    注意: 这是对接任务，训练目标是距离矩阵，不是 logKa。"""
    def __init__(self, data_list):
        # data_list: list of (prot_feats, lig_feats, prot_coords, lig_coords, dist_matrix)
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        prot_f, lig_f, prot_c, lig_c, dist_mat = self.data[idx]
        return (torch.FloatTensor(prot_f),       # (N_p, 10)
                torch.FloatTensor(lig_f),        # (N_l, 10)
                torch.FloatTensor(prot_c),       # (N_p, 3)
                torch.FloatTensor(lig_c),        # (N_l, 3)
                torch.FloatTensor(dist_mat))     # (N_p, N_l)


# ---- 随机 80/20 划分训练/测试集 ----
indices = np.random.permutation(len(all_data))
split = int(0.8 * len(all_data))
train_idx, test_idx = indices[:split], indices[split:]

train_data = [all_data[i] for i in train_idx]
test_data = [all_data[i] for i in test_idx]

train_loader = DataLoader(TankBindDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TankBindDataset(test_data), batch_size=BATCH_SIZE, shuffle=False)

# 展示数据集划分信息
split_df = pd.DataFrame({
    '数据集': ['训练集', '测试集', '总计'],
    '样本数': [len(train_data), len(test_data), len(all_data)],
    '比例': [f'{len(train_data)/len(all_data)*100:.0f}%',
            f'{len(test_data)/len(all_data)*100:.0f}%', '100%']
})
display(split_df)

## 4. 模型架构

ToyTankBind 模型由以下关键组件构成：

### 三角更新模块 (TriangleUpdate)
灵感来自 AlphaFold2 的三角乘法更新。核心思想：蛋白原子 $i$ 到配体原子 $j$ 的距离表示 $z_{ij}$，应当与经过中间节点 $k$ 的路径 $(z_{ik}, z_{kj})$ 保持几何一致性。

对矩形 ($N_p \neq N_l$) 距离矩阵的分解三角更新：
$$\text{left\_agg}[i] = \frac{1}{N_l}\sum_k \sigma(W_L \cdot z[i,k])$$
$$\text{right\_agg}[j] = \frac{1}{N_p}\sum_k \sigma(W_R \cdot z[k,j])$$
$$z[i,j] \leftarrow z[i,j] + W_O(\text{left\_agg}[i] \odot \text{right\_agg}[j])$$

### 整体流程
1. 蛋白和配体原子各自通过线性嵌入层映射到隐空间
2. 外积构建蛋白-配体对的表示矩阵 $z_{ij} = h_i^{prot} + h_j^{lig}$
3. 多层三角更新增强几何一致性
4. 距离预测头将隐表示映射为距离值

In [ ]:
class TriangleUpdate(nn.Module):
    """三角更新模块（简化版）

    灵感来自 AlphaFold2 的三角乘法更新 (triangle multiplicative update)。
    核心思想：蛋白原子 i 到配体原子 j 的距离表示 z_ij，
    应当与经过中间节点 k 的路径 (z_ik, z_kj) 保持几何一致性。

    更新规则:
      z_ij += Linear(Sigma_k z_ik * z_kj)

    这里简化为：沿蛋白维度 (outgoing) 做三角乘法更新。
    在完整的 TankBind 中，三角更新确保预测的距离矩阵满足三角不等式。
    """
    def __init__(self, hidden_dim):
        super().__init__()
        # 投影层：将 z 投影到 "左" 和 "右" 分量，用于三角乘法
        self.left_proj = nn.Linear(hidden_dim, hidden_dim)
        self.right_proj = nn.Linear(hidden_dim, hidden_dim)
        # 输出投影 + 归一化
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.layer_norm = nn.LayerNorm(hidden_dim)

    def forward(self, z):
        """
        参数:
          z : (N_p, N_l, hidden_dim)  蛋白-配体对的隐表示矩阵

        返回:
          z_updated : (N_p, N_l, hidden_dim)  更新后的表示

        对矩形(N_p != N_l)距离矩阵的分解三角更新：
          left[i,k]  = sigmoid(left_proj(z[i,k]))     -- 门控投影
          right[k,j] = sigmoid(right_proj(z[k,j]))    -- 门控投影
          left_agg[i]  = sum_k left[i,k]              -- 蛋白原子聚合配体信息
          right_agg[j] = sum_k right[k,j]             -- 配体原子聚合蛋白信息
          tri[i,j]     = left_agg[i] * right_agg[j]   -- 外积组合
          z[i,j]       = z[i,j] + out_proj(tri[i,j])  -- 残差连接
        """
        z_normed = self.layer_norm(z)

        # 门控投影: (N_p, N_l, hidden) -> (N_p, N_l, hidden)
        left = torch.sigmoid(self.left_proj(z_normed))
        right = torch.sigmoid(self.right_proj(z_normed))

        # 分解三角更新 (适用于矩形 N_p x N_l 距离矩阵):
        # 标准三角乘法 (sum_k left[i,k]*right[k,j]) 要求方阵 (N_p==N_l)，
        # 此处改用分解形式：分别沿配体和蛋白维度聚合，再外积组合
        left_agg = left.mean(dim=1)   # (N_p, hidden) -- 每个蛋白原子聚合所有配体信息
        right_agg = right.mean(dim=0) # (N_l, hidden) -- 每个配体原子聚合所有蛋白信息
        tri = left_agg.unsqueeze(1) * right_agg.unsqueeze(0)  # (N_p, N_l, hidden)

        # 输出投影并残差连接
        return z + self.out_proj(tri)


class ToyTankBind(nn.Module):
    """
    TankBind 玩具模型 -- 基于距离矩阵预测的分子对接。

    核心架构：
      1. 蛋白和配体原子各自通过线性嵌入层映射到隐空间；
      2. 外积构建蛋白-配体对的表示矩阵 z_ij = prot_h_i + lig_h_j；
      3. 多层三角更新增强几何一致性；
      4. 距离预测头将隐表示映射为距离值。

    与 AlphaFold2 的联系：
      三角更新模块借鉴了 AlphaFold2 中 pair representation 的
      三角乘法更新思想，确保预测的距离矩阵满足三角不等式等几何约束。

    与传统打分模型 (IGN/PIGNet/RTMScore) 的区别：
      - 打分模型预测标量 logKa（结合亲和力）
      - TankBind 预测 N_p x N_l 距离矩阵（空间几何信息）
      - 距离矩阵可进一步用于恢复配体 3D 坐标（对接）
    """
    def __init__(self, atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM,
                 n_layers=N_TRIANGLE_LAYERS, max_dist=MAX_DIST):
        super().__init__()
        self.max_dist = max_dist

        # 1. 蛋白/配体原子嵌入层（分开嵌入，学习各自的特征空间）
        self.prot_embed = nn.Linear(atom_dim, hidden_dim)
        self.lig_embed = nn.Linear(atom_dim, hidden_dim)

        # 2. 多层三角更新（增强距离矩阵的几何一致性）
        self.triangle_layers = nn.ModuleList([
            TriangleUpdate(hidden_dim) for _ in range(n_layers)
        ])

        # 3. 距离预测头: hidden -> 1，输出经 Sigmoid 缩放到 [0, max_dist]
        self.dist_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()            # 输出 [0, 1]，后乘 max_dist
        )

    def forward(self, prot_feats, lig_feats):
        """
        参数:
          prot_feats : (N_p, atom_dim)  蛋白口袋原子特征
          lig_feats  : (N_l, atom_dim)  配体原子特征

        返回:
          pred_dist  : (N_p, N_l)       预测的蛋白-配体距离矩阵
        """
        # Step 1: 原子嵌入
        prot_h = self.prot_embed(prot_feats)    # (N_p, hidden)
        lig_h = self.lig_embed(lig_feats)       # (N_l, hidden)

        # Step 2: 构建 pair 表示矩阵
        # z[i,j] = prot_h[i] + lig_h[j]，通过广播实现
        # 这相当于外积+求和，编码了蛋白原子 i 与配体原子 j 的联合信息
        z = prot_h.unsqueeze(1) + lig_h.unsqueeze(0)   # (N_p, N_l, hidden)

        # Step 3: 三角更新 -- 传播几何约束
        # 每一层三角更新让 z_ij 融合来自 z_ik 和 z_kj 的信息
        for tri_layer in self.triangle_layers:
            z = tri_layer(z)

        # Step 4: 距离预测
        # dist_head 输出 (N_p, N_l, 1)，squeeze 后乘 max_dist 得到距离值
        pred_dist = self.dist_head(z).squeeze(-1) * self.max_dist  # (N_p, N_l)

        return pred_dist

In [ ]:
# ---- 坐标恢复函数 ----

@torch.enable_grad()
def recover_coords_gradient_descent(pred_dist, prot_coords, n_lig,
                                    n_steps=COORD_RECOVERY_STEPS,
                                    lr=COORD_RECOVERY_LR):
    """从预测距离矩阵恢复配体坐标（参考 TankBind 原始 generation_utils.py）。

    核心思想：
      给定预测的蛋白-配体距离矩阵和已知的蛋白坐标，
      通过梯度下降找到一组配体坐标，使得 cdist(prot, lig) 尽量接近 pred_dist。

    算法流程：
      1. 在蛋白口袋质心附近随机初始化配体原子坐标（参考原始实现的初始化方式）
      2. 计算当前配体坐标与蛋白坐标之间的距离矩阵
      3. 用 Adam 优化器最小化 |cdist(prot, lig) - pred_dist|^2
      4. 经过足够步数后返回优化后的坐标

    参数:
      pred_dist   : (N_p, N_l) numpy 数组，预测的蛋白-配体距离矩阵
      prot_coords : (N_p, 3)   numpy 数组，蛋白原子坐标
      n_lig       : int        配体原子数
      n_steps     : int        梯度下降步数
      lr          : float      学习率

    返回:
      lig_coords  : (N_l, 3) numpy 数组，恢复的配体坐标
    """
    # 转为 GPU tensor 加速梯度下降
    device = DEVICE
    pred_dist_t = torch.FloatTensor(pred_dist).to(device)      # (N_p, N_l)
    prot_coords_t = torch.FloatTensor(prot_coords).to(device)  # (N_p, 3)

    # 初始化: 在口袋质心附近随机扰动（参考原始实现 5*(2*rand-1)+center）
    center = prot_coords_t.mean(dim=0)  # (3,)
    lig_coords_t = 5.0 * (2.0 * torch.rand(n_lig, 3, device=device) - 1.0) + center.unsqueeze(0)
    lig_coords_t = nn.Parameter(lig_coords_t)  # 设为可优化参数

    opt = torch.optim.Adam([lig_coords_t], lr=lr)

    for _ in range(n_steps):
        opt.zero_grad()
        # 计算当前配体坐标与蛋白坐标的距离矩阵: (N_p, N_l)
        current_dist = torch.cdist(prot_coords_t.unsqueeze(0),
                                   lig_coords_t.unsqueeze(0)).squeeze(0)
        # 截断距离（与训练时一致）
        current_dist_clamped = torch.clamp(current_dist, max=MAX_DIST)
        # 损失: MSE(当前距离, 预测距离)
        loss = ((current_dist_clamped - pred_dist_t) ** 2).mean()
        loss.backward()
        opt.step()

    return lig_coords_t.detach().cpu().numpy()


# ---- 实例化模型并展示参数信息 ----
model = ToyTankBind(atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

model_info = pd.DataFrame({
    '信息': ['模型名称', '总参数量', '可训练参数量', '设备'],
    '值': ['ToyTankBind', f'{total_params:,}', f'{trainable_params:,}', str(DEVICE)]
})
display(model_info)

print(f"\n模型结构:\n{model}")

## 5. 训练

训练目标：最小化预测距离矩阵与真实距离矩阵之间的 MSE 损失。

$$\mathcal{L} = \text{MSE}(\hat{D}, D) = \frac{1}{N_p \cdot N_l} \sum_{i,j} (\hat{d}_{ij} - d_{ij})^2$$

其中 $\hat{D}$ 为预测距离矩阵，$D$ 为真实距离矩阵。

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

print(f"开始训练 {N_EPOCHS} 轮...\n")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    train_losses = []

    for prot_f, lig_f, prot_c, lig_c, dist_mat in train_loader:
        # 每个 batch 只有 1 个样本 (变长结构)，去掉 batch 维度
        prot_f = prot_f.squeeze(0).to(DEVICE)       # (N_p, 10)
        lig_f = lig_f.squeeze(0).to(DEVICE)         # (N_l, 10)
        dist_mat = dist_mat.squeeze(0).to(DEVICE)   # (N_p, N_l)

        optimizer.zero_grad()
        pred_dist = model(prot_f, lig_f)            # (N_p, N_l)
        loss = criterion(pred_dist, dist_mat)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # ---- 每 20 轮评估一次 ----
    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        val_losses = []
        with torch.no_grad():
            for prot_f, lig_f, prot_c, lig_c, dist_mat in test_loader:
                prot_f = prot_f.squeeze(0).to(DEVICE)
                lig_f = lig_f.squeeze(0).to(DEVICE)
                dist_mat = dist_mat.squeeze(0).to(DEVICE)
                pred_dist = model(prot_f, lig_f)
                val_losses.append(criterion(pred_dist, dist_mat).item())

        avg_train = np.mean(train_losses) if train_losses else float('nan')
        avg_val = np.mean(val_losses) if val_losses else float('nan')
        print(f"  Epoch {epoch:>4d}/{N_EPOCHS}  |  Train Loss: {avg_train:.4f}  |  Val Loss: {avg_val:.4f}")

## 6. 评估与可视化

评估流程：
1. 对每个测试样本预测距离矩阵
2. 通过梯度下降从预测距离矩阵恢复配体 3D 坐标
3. 计算恢复坐标与真实坐标之间的 RMSD

评估指标：
- **距离矩阵 Pearson R**: 预测距离与真实距离的相关性
- **RMSD (Root Mean Square Deviation)**: 恢复坐标与真实坐标的均方根偏差
- **RMSD < 2A / 5A 比例**: 对接成功率的常用阈值

In [ ]:
print("在测试集上评估...")
print("对每个测试样本进行距离矩阵预测 + 梯度下降坐标恢复（可能需要几分钟）...")

model.eval()

# 收集距离矩阵预测相关性和 RMSD
all_true_dists = []     # 扁平化的真实距离值
all_pred_dists = []     # 扁平化的预测距离值
all_rmsds = []          # 每个复合物的坐标恢复 RMSD

with torch.no_grad():
    for i, (prot_f, lig_f, prot_c, lig_c, dist_mat) in enumerate(test_loader):
        prot_f = prot_f.squeeze(0).to(DEVICE)       # (N_p, 10)
        lig_f = lig_f.squeeze(0).to(DEVICE)         # (N_l, 10)
        prot_c = prot_c.squeeze(0).numpy()          # (N_p, 3)
        lig_c = lig_c.squeeze(0).numpy()            # (N_l, 3)
        dist_mat = dist_mat.squeeze(0)              # (N_p, N_l)

        # 预测距离矩阵
        pred_dist = model(prot_f, lig_f).cpu()      # (N_p, N_l)

        # 收集扁平化距离值用于相关性分析
        all_true_dists.append(dist_mat.numpy().flatten())
        all_pred_dists.append(pred_dist.numpy().flatten())

        # 梯度下降恢复配体坐标并计算 RMSD
        n_lig = lig_c.shape[0]
        recovered_coords = recover_coords_gradient_descent(
            pred_dist.numpy(), prot_c, n_lig,
            n_steps=COORD_RECOVERY_STEPS, lr=COORD_RECOVERY_LR
        )
        rmsd = compute_rmsd(recovered_coords, lig_c)
        all_rmsds.append(rmsd)

        if (i + 1) % 10 == 0:
            print(f"  评估进度: {i + 1}/{len(test_loader)}")

# 合并所有距离值
all_true_dists = np.concatenate(all_true_dists)
all_pred_dists = np.concatenate(all_pred_dists)
all_rmsds = np.array(all_rmsds)

# ---- 计算指标 ----
dist_r, _ = pearsonr(all_true_dists, all_pred_dists)
mean_rmsd = np.mean(all_rmsds)
median_rmsd = np.median(all_rmsds)
pct_under_2 = np.mean(all_rmsds < 2.0) * 100      # RMSD < 2A 的比例
pct_under_5 = np.mean(all_rmsds < 5.0) * 100      # RMSD < 5A 的比例

# 以 DataFrame 展示结果
results_df = pd.DataFrame({
    '指标': ['样本数', '距离矩阵 Pearson R', '平均 RMSD', '中位数 RMSD',
            'RMSD < 2A', 'RMSD < 5A'],
    '值': [f'{len(all_rmsds)}', f'{dist_r:.4f}', f'{mean_rmsd:.4f} A',
           f'{median_rmsd:.4f} A', f'{pct_under_2:.1f}%', f'{pct_under_5:.1f}%']
})
display(results_df)

In [ ]:
# ---- 可视化：两张子图 ----
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# 子图 1: 距离矩阵散点图 (采样展示，避免点太多)
ax1 = axes[0]
n_sample = min(5000, len(all_true_dists))
sample_idx = np.random.choice(len(all_true_dists), n_sample, replace=False)
ax1.scatter(all_true_dists[sample_idx], all_pred_dists[sample_idx],
            alpha=0.3, s=8, edgecolors='none', c='steelblue')
lo = 0
hi = MAX_DIST + 1
ax1.plot([lo, hi], [lo, hi], 'r--', linewidth=1.0, label='y = x')
ax1.set_xlabel('True Distance (A)', fontsize=12)
ax1.set_ylabel('Predicted Distance (A)', fontsize=12)
ax1.set_title(f'Distance Matrix Correlation  |  R={dist_r:.3f}', fontsize=13)
ax1.legend(loc='upper left')
ax1.set_xlim(lo, hi)
ax1.set_ylim(lo, hi)
ax1.set_aspect('equal')

# 子图 2: RMSD 直方图
ax2 = axes[1]
ax2.hist(all_rmsds, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax2.axvline(mean_rmsd, color='red', linestyle='--', linewidth=1.5,
            label=f'Mean={mean_rmsd:.2f} A')
ax2.axvline(median_rmsd, color='orange', linestyle='--', linewidth=1.5,
            label=f'Median={median_rmsd:.2f} A')
ax2.set_xlabel('RMSD (A)', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Ligand Coordinate Recovery RMSD', fontsize=13)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 总结

本教程实现了 TankBind 的简化版本，展示了基于距离矩阵预测的分子对接方法的核心思想。

### 关键要点

1. **对接 vs 打分**: 打分模型预测标量结合亲和力 ($\log K_a$)，对接模型预测 $N_p \times N_l$ 距离矩阵，包含完整的空间几何信息。

2. **三角更新模块**: 借鉴 AlphaFold2 的三角乘法更新思想，通过传播 $z_{ik}$ 和 $z_{kj}$ 的信息来增强 $z_{ij}$ 的几何一致性，确保预测距离矩阵满足三角不等式。

3. **梯度下降坐标恢复**: 给定预测距离矩阵和蛋白坐标，通过优化配体坐标使 $\text{cdist}(\text{prot}, \text{lig}) \approx \hat{D}$ 来恢复配体 3D 构象。

4. **评估指标**: RMSD 是对接质量的标准评估指标，RMSD < 2A 通常被视为成功对接。

### 简化说明

本教程为教学目的做了以下简化：
- 使用简单的 10 维原子特征（完整版使用更丰富的蛋白/配体描述符）
- 使用分解形式的三角更新（完整版包含 outgoing 和 incoming 两种三角更新）
- 训练数据为 CASF-2016 核心集 285 个复合物（完整版使用 PDBbind 全集）
- 口袋原子通过距离截断获取（完整版使用 P2Rank 等方法预测结合位点）